# urllib3: 5 Basic Usage Examples

In [ ]:
# Example 1: Basic GET request
import urllib3
http = urllib3.PoolManager()
r = http.request('GET', 'https://httpbingo.org/get')
print(r.status, r.data[:60])
assert r.status == 200, (r.status, r.data[:300])
assert r.headers['Content-Type'].startswith('application/json'), r.headers
body = r.json()
assert body['url'] == 'https://httpbingo.org/get' and body['method'] == 'GET' and body['args'] == {}, body


In [ ]:
# Example 2: Custom headers
import urllib3
http = urllib3.PoolManager()
r = http.request('GET', 'https://httpbingo.org/headers', headers={'X-Test': 'true'})
print(r.status, r.data[:60])
assert r.status == 200, (r.status, r.data[:300])
echoed = r.json()['headers']
assert echoed.get('X-Test') == ['true'], echoed  # the server echoes the headers urllib3 sent


In [ ]:
# Example 3: Timeout
# A 1 s delay answers inside a 10 s read timeout; a 3 s delay inside a 0.5 s read timeout raises ReadTimeoutError.
import urllib3
http = urllib3.PoolManager(timeout=urllib3.util.Timeout(connect=10.0, read=10.0))
r = http.request('GET', 'https://httpbingo.org/delay/1')
print(r.status)
assert r.status == 200, (r.status, r.data[:300])
try:
    http.request('GET', 'https://httpbingo.org/delay/3', timeout=urllib3.util.Timeout(connect=10.0, read=0.5), retries=False)
except urllib3.exceptions.ReadTimeoutError as e:
    print('timed out as expected:', type(e).__name__)
else:
    raise AssertionError('a 3 s delay came back inside a 0.5 s read timeout')


In [ ]:
# Example 4: Retry
# status_forcelist makes a 500 retryable; raise_on_status=False returns the last response after the retries run out.
import urllib3
from urllib3.util.retry import Retry
http = urllib3.PoolManager(retries=Retry(total=2, status_forcelist=[500], backoff_factor=0, raise_on_status=False))
r = http.request('GET', 'https://httpbingo.org/status/500')
print(r.status, r.retries.history)
assert r.status == 500, r.status
assert [h.status for h in r.retries.history] == [500, 500], r.retries.history  # two retries, both a 500


In [ ]:
# Example 5: Using a proxy -- the cell runs its own loopback forward proxy, so nothing leaves the box.
# For a plain-http target ProxyManager sends the ABSOLUTE URL as the request target (that is what a
# forward proxy reads); this proxy answers with what it received instead of forwarding it.
import json, threading
from http.server import BaseHTTPRequestHandler, HTTPServer
import urllib3

class EchoProxy(BaseHTTPRequestHandler):
    def do_GET(self):
        body = json.dumps({"target": self.path, "host": self.headers["Host"]}).encode()
        self.send_response(200); self.send_header("Content-Length", str(len(body))); self.end_headers()
        self.wfile.write(body)
    def log_message(self, *args): pass

server = HTTPServer(("127.0.0.1", 0), EchoProxy)
threading.Thread(target=server.serve_forever, daemon=True).start()
proxy = urllib3.ProxyManager(f"http://127.0.0.1:{server.server_port}/")
try:
    r = proxy.request('GET', 'http://example.invalid/ip')  # .invalid never resolves: only the proxy can answer
    print(r.status, r.data[:60])
    assert r.status == 200 and r.json() == {"target": "http://example.invalid/ip", "host": "example.invalid"}, r.data
finally:
    proxy.clear(); server.shutdown(); server.server_close()
